# Visualize traffic spikes

This notebook walks through the analyzer's pipeline: create access-log lines, parse timestamps, count requests in fixed windows, detect unusually busy windows, and plot the result. The sample is deterministic and includes one injected burst, so it runs without a separate log file.

To try your own data, replace the sample-log cell with code that reads your Apache/Nginx log into `lines`.

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

# Make the package importable when running from either the repository root or notebooks/.
repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
sys.path.insert(0, str(repo_root / "src"))

from poisson_log_analyzer import bin_counts, detect_spikes, parse_lines

## Create a sample access log

Each generated line uses the supported combined-log timestamp format. Most one-minute windows have a Poisson-distributed request count; one window has a deliberately larger burst.

In [ ]:
rng = np.random.default_rng(42)
window_seconds = 60
window_count = 60
baseline_requests_per_window = 8
spike_index = 30
spike_requests = 60
start_time = datetime(2025, 1, 1, tzinfo=timezone.utc)

sample_counts = rng.poisson(baseline_requests_per_window, size=window_count)
sample_counts[spike_index] = spike_requests

lines = []
for minute, request_count in enumerate(sample_counts):
    seconds = rng.integers(0, window_seconds, size=int(request_count))
    for second in seconds:
        timestamp = start_time + timedelta(minutes=minute, seconds=int(second))
        formatted_time = timestamp.strftime("%d/%b/%Y:%H:%M:%S %z")
        lines.append(
            f'192.0.2.1 - - [{formatted_time}] "GET /demo HTTP/1.1" 200 512'
        )

print(f"Generated {len(lines)} access-log lines across {window_count} minutes.")
print(lines[0])

## Parse, count, and detect

`parse_lines` skips malformed lines, `bin_counts` includes empty windows between the first and last timestamp, and `detect_spikes` estimates a baseline rate and applies Benjamini–Hochberg correction.

In [ ]:
entries = list(parse_lines(lines))
start, counts = bin_counts(
    (entry.timestamp for entry in entries), window_seconds=window_seconds
)
result = detect_spikes(counts, alpha=0.01)

print(f"Parsed requests: {len(entries)}")
print(f"Windows: {len(counts)} ({window_seconds} seconds each)")
print(f"Baseline rate: {result.rate:.2f} requests/window")
print(f"Baseline dispersion: {result.dispersion:.2f}")
print(f"Flagged windows: {result.flagged}")
[(i, int(counts[i]), float(result.q_values[i])) for i in result.flagged]

## Visualize the result

The line shows requests per window; red markers identify windows the detector flagged as spikes. The dashed line is the fitted baseline request rate.

In [ ]:
minutes = np.arange(len(counts)) * window_seconds / 60
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(minutes, counts, color="steelblue", marker="o", markersize=3, label="Requests/window")
ax.axhline(result.rate, color="gray", linestyle="--", label="Fitted baseline")
if result.flagged:
    flagged = np.asarray(result.flagged)
    ax.scatter(
        minutes[flagged], counts[flagged],
        color="crimson", zorder=3, label="Detected spike",
    )
ax.set(title="Request volume and detected spikes", xlabel="Minutes from start", ylabel="Requests per minute")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

A dispersion value substantially above 1 suggests the baseline counts vary more than a Poisson model expects; interpret spike p-values cautiously in that case. You can experiment with the generated baseline, burst size, window width, and `alpha` above.